In [1]:
import gc
import os
import time
import numpy as np
import pandas as pd
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import mlflow


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_proc = psutil.Process(os.getpid())

In [3]:
# 1. Define the dataset
PROMPTS = [
    ("simple_qa_1",   "What is the capital of Japan?", 32),
    ("simple_qa_2",   "Who wrote the play 'Hamlet'?", 32),
    ("reasoning_1",   "If a train travels 60 km in 45 minutes, what is its average speed in km/h? Explain.", 128),
    ("reasoning_2",   "A bat and a ball cost $1.10 together. The bat costs $1.00 more than the ball. How much is the ball? Reason step by step.", 128),
    ("coding_1",      "Write a Python function that returns the n-th Fibonacci number using memoization.", 200),
    ("coding_2",      "Write a SQL query to find the second highest salary from an Employees table.", 128),
    ("summarization", "Summarize the following in one sentence: Quantization reduces the numerical precision of model weights and activations, lowering memory and bandwidth needs while usually keeping accuracy close to the full-precision baseline.", 64),
    ("long_context",  "Here are five facts: (1) The sky is blue. (2) Water boils at 100C at sea level. (3) Paris is in France. (4) Honey does not spoil. (5) Spiders have eight legs. Which fact is about cooking, and which is about geography?", 96),
    ("math_1",        "Compute 17 * 23 and show the multiplication steps.", 96),
    ("math_2",        "What is the derivative of f(x) = 3x^2 + 2x - 5 with respect to x?", 64),
]

dataset = pd.DataFrame(PROMPTS, columns=["name", "prompt", "expected_len"])

In [4]:
MODEL_FP16 = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_AWQ  = "./models/llama-3.2-3b-instruct-awq-custom"
MODEL_GPTQ = "./models/llama-3.2-3b-instruct-gptq-custom"

# Metrics

In [5]:
def gpu_allocated_gb(device=DEVICE): return torch.cuda.memory_allocated() / 1024**3 if device == "cuda" else 0.0
def gpu_reserved_gb(device=DEVICE): return torch.cuda.memory_reserved() / 1024**3 if device == "cuda" else 0.0
def gpu_peak_reserved_gb(device=DEVICE): return torch.cuda.max_memory_reserved() / 1024**3 if device == "cuda" else 0.0
def gpu_peak_gb(device=DEVICE): return torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0.0

def cpu_resident_gb(): return _proc.memory_info().rss / 1024**3
def cpu_virtual_gb(): return _proc.memory_info().vms / 1024**3

## Local Logging

In [6]:
metric_df = pd.DataFrame(
    columns=["LLAMA_3_2_3B_FP16", "LLAMA_3_2_3B_AWQ_CUSTOM","LLAMA_3_2_3B_GPTQ_CUSTOM"],
    index=pd.MultiIndex.from_tuples([], names=["metrics", "stage"])
)

In [7]:
def add_metric(df, metric, stage, model, value):
    # Create row if metric doesn't exist
    key = (metric, stage)
    if key not in df.index:
        df.loc[key, :] = [None] * len(df.columns)

    # Update the appropriate model column
    df.loc[key, model] = value
    return df

## MLFlow Logging

In [8]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("AWQ Quantization Benchmarking")
print(mlflow.get_tracking_uri())

http://localhost:5000


In [9]:
def mlflow_start_run(name: str):
    if mlflow.active_run():
        mlflow.end_run()
    return mlflow.start_run(run_name=name)

# Utils

In [10]:
def gpu_reset_peak(device=DEVICE): torch.cuda.reset_peak_memory_stats() if device == "cuda" else None

In [11]:
def clear_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# Timers

In [12]:
class BenchmarkTimer:
    def __init__(self, sync=True):
        self.sync = sync
        self.elapsed = None
    def __enter__(self):
        if self.sync and torch.cuda.is_available(): torch.cuda.synchronize()
        self._start = time.perf_counter()
        return self
    def __exit__(self, *exc):
        if self.sync and torch.cuda.is_available(): torch.cuda.synchronize()
        self.elapsed = time.perf_counter() - self._start
        return False

# Inference

## Custom TTFT Streamer

In [13]:
import time

class TTFTStreamer:
    """A minimal streamer designed solely to capture Time to First Token."""
    def __init__(self):
        self.start_time = None
        self.ttft = None
        self._call_count = 0

    def put(self, value):
        """Called by .generate() to push new tokens."""
        self._call_count += 1
        
        # Call 1: The input prompt
        # Call 2: The first newly generated token
        if self._call_count == 2 and self.start_time is not None:
            self.ttft = time.perf_counter() - self.start_time

    def end(self):
        """Called by .generate() to signal the end of generation."""
        pass

## Inference Definition

In [14]:
def run_inference(model_id, df, model_label):

    mlflow_start_run(f"Benchmarking {model_label}")

    print(f"\n==================================================")
    print(f"=== BENCHMARKING MODEL: {model_label} ===")
    print(f"==================================================")

    # --- CHECKPOINT 1: BASELINE (BEFORE LOADING) ---
    clear_cache()
    gpu_reset_peak()

    base_gpu = gpu_allocated_gb()
    base_cpu = cpu_resident_gb()
    print(f"[Baseline] GPU Allocated: {base_gpu:.3f} GB | CPU Resident: {base_cpu:.3f} GB")

    add_metric(metric_df, "gpu_allocated", "baseline", model_label, base_gpu)
    add_metric(metric_df, "cpu_resident", "baseline", model_label, base_cpu)
    mlflow.log_metric("baseline.gpu_allocated_gb", base_gpu)
    mlflow.log_metric("baseline.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("baseline.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("baseline.cpu_resident_gb", base_cpu)
    mlflow.log_metric("baseline.cpu_virtual_gb", cpu_virtual_gb())

    print(f"\n=== Loading Model: {model_label} ({model_id}) ===")
    
    with BenchmarkTimer() as load_timer:
    # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        # Llama 3 models require setting the pad token if it's not set
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            
        # Load model (Transformers natively supports AWQ if autoawq is installed)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )


    # --- CHECKPOINT 2: POST-LOAD METRICS ---
    post_load_gpu = gpu_allocated_gb()
    reported_model_size_gb = (post_load_gpu - base_gpu) # Calculated delta via VRAM allocation
    
    print(f"[Loaded] Model Load Time: {load_timer.elapsed:.2f} seconds")
    print(f"[Loaded] Measured Model Weight Size: {reported_model_size_gb:.3f} GB")
    print(f"[Loaded] Current GPU Allocated: {post_load_gpu:.3f} GB | GPU Reserved: {gpu_reserved_gb():.3f} GB")

    add_metric(metric_df, "load_time", "post_load", model_label, load_timer.elapsed)
    add_metric(metric_df, "gpu_allocated", "post_load", model_label, post_load_gpu)
    add_metric(metric_df, "model_size_gb", "post_load", model_label, reported_model_size_gb)

    mlflow.log_metric("post_load.load_time_seconds", load_timer.elapsed)
    mlflow.log_metric("post_load.gpu_allocated_gb", post_load_gpu)
    mlflow.log_metric("post_load.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("post_load.gpu_peak_reserved_gb", gpu_peak_reserved_gb())
    mlflow.log_metric("post_load.reported_model_size_gb", reported_model_size_gb)

    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    total_size_gb = (param_size + buffer_size) / (1024 ** 3)
    
    
    print(f"-> Model Size in Memory: {total_size_gb:.2f} GB")

    add_metric(metric_df, "model_size_memory_gb", "post_load", model_label, total_size_gb)
    mlflow.log_metric("post_load.model_size_memory_gb", total_size_gb)

    results = []
    latencies = []
    ttft_list = []
    tokens_per_sec_list = []
    
    print(f"Running inference on {len(df)} samples...")
    for idx, row in df.iterrows():
        # Format the prompt using Llama 3's chat template for optimal Instruct performance
        messages = [{"role": "user", "content": row["prompt"]}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        
        # Instantiate the custom streamer for this specific prompt
        streamer = TTFTStreamer()
        # Generate text
        with BenchmarkTimer() as inf_timer:
            with torch.no_grad():
                # outputs = model.generate(
                #     **inputs,
                #     max_new_tokens=int(row["expected_len"]),
                #     do_sample=False, # Deterministic greedy decoding for benchmarking
                #     pad_token_id=tokenizer.eos_token_id
                # )
                # Start the TTFT clock exactly when generation starts
                streamer.start_time = time.perf_counter() 
                
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=int(row["expected_len"]),
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    streamer=streamer  # <--- INJECT STREAMER HERE
                )
        
        # Decode only the newly generated tokens
        generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]
        num_generated_tokens = len(generated_tokens)

        # If the model fails to generate anything, gracefully handle the NoneType
        ttft_seconds = streamer.ttft if streamer.ttft is not None else 0.0
        ttft_list.append(ttft_seconds)  # Store TTFT for this specific prompt
        # Calculate tokens per second (excluding the TTFT wait time for accuracy)
        generation_time = inf_timer.elapsed - ttft_seconds
        tokens_per_sec = (num_generated_tokens - 1) / generation_time if generation_time > 0 else 0

        # tokens_per_sec = num_generated_tokens / inf_timer.elapsed if inf_timer.elapsed > 0 else 0
        tokens_per_sec_list.append(tokens_per_sec)

        response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(response)
        latencies.append(inf_timer.elapsed)
        print(f"[{row['name']}] Done.")

        print(f"  -> [{row['name']}] Generated {num_generated_tokens} tokens in {inf_timer.elapsed:.3f}s ({tokens_per_sec:.2f} tok/s)")

        
 

        
    # --- CHECKPOINT 4: POST-INFERENCE / PEAK ANALYSIS ---
    peak_gpu = gpu_peak_gb()
    print(f"\n[Post-Inference Summary]")
    print(f"[*] Max Peak GPU Memory During Run: {peak_gpu:.3f} GB")
    print(f"[*] Total Execution VRAM Overhead (Peak - Baseline): {peak_gpu - base_gpu:.3f} GB")
    print(f"[*] Average Generation Latency: {sum(latencies)/len(latencies):.3f} seconds")

    add_metric(metric_df, "average_inference_latency", "post_inference", model_label, np.mean(latencies))
    add_metric(metric_df, "average_tokens_per_sec", "post_inference", model_label, np.mean(tokens_per_sec_list))
    add_metric(metric_df, "peak_gpu_allocated", "post_inference", model_label, peak_gpu)

    mlflow.log_metric("post_inference.ttft_average_seconds", np.mean(ttft_list))
    mlflow.log_metric("post_inference.average_inference_latency_seconds", np.mean(latencies))
    mlflow.log_metric("post_inference.average_tokens_per_sec", np.mean(tokens_per_sec_list))
    mlflow.log_metric("post_inference.peak_gpu_allocated_gb", peak_gpu)
    mlflow.log_metric("post_inference.total_execution_vram_overhead_gb", peak_gpu - base_gpu)
    mlflow.log_metric("post_inference.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("post_inference.gpu_peak_reserved_gb", gpu_peak_reserved_gb())

    # Clean up GPU memory before the next model loads
    del model
    del tokenizer
    # torch.cuda.empty_cache()
    clear_cache()
    
    final_gpu = gpu_allocated_gb()
    print(f"[Cleanup] Post-Cleanup GPU Allocated: {final_gpu:.3f} GB (Should match or be close to Baseline)")
    add_metric(metric_df, "gpu_allocated", "post_cleanup", model_label, final_gpu)
    mlflow.log_metric("post_cleanup.gpu_allocated_gb", final_gpu)
    mlflow.log_metric("post_cleanup.gpu_reserved_gb", gpu_reserved_gb())
    mlflow.log_metric("post_cleanup.gpu_peak_reserved_gb", gpu_peak_reserved_gb())

    print(f"\n=== Finished Benchmarking Model: {model_label} ===\n")
    print(f"==================================================\n")
    print(f"=== METRICS DATAFRAME ===")
    print(metric_df)

    return results

In [65]:
# 3. Run evaluation for both models
dataset["response_fp16"] = run_inference(MODEL_FP16, dataset, "LLAMA_3_2_3B_FP16")


=== BENCHMARKING MODEL: LLAMA_3_2_3B_FP16 ===
[Baseline] GPU Allocated: 2.106 GB | CPU Resident: 2.881 GB

=== Loading Model: LLAMA_3_2_3B_FP16 (meta-llama/Llama-3.2-3B-Instruct) ===


INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-3.2-3B-Instruct "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"


[Loaded] Model Load Time: 3.32 seconds
[Loaded] Measured Model Weight Size: 5.984 GB
[Loaded] Current GPU Allocated: 8.090 GB | GPU Reserved: 8.109 GB
-> Model Size in Memory: 5.98 GB
Running inference on 10 samples...
[simple_qa_1] Done.
  -> [simple_qa_1] Generated 8 tokens in 0.092s (90.26 tok/s)
[simple_qa_2] Done.
  -> [simple_qa_2] Generated 13 tokens in 0.146s (90.61 tok/s)
[reasoning_1] Done.
  -> [reasoning_1] Generated 86 tokens in 0.957s (90.10 tok/s)
[reasoning_2] Done.
  -> [reasoning_2] Generated 128 tokens in 1.430s (89.71 tok/s)
[coding_1] Done.
  -> [coding_1] Generated 200 tokens in 2.223s (90.05 tok/s)
[coding_2] Done.
  -> [coding_2] Generated 128 tokens in 1.421s (90.26 tok/s)
[summarization] Done.
  -> [summarization] Generated 29 tokens in 0.327s (89.63 tok/s)
[long_context] Done.
  -> [long_context] Generated 43 tokens in 0.481s (90.26 tok/s)
[math_1] Done.
  -> [math_1] Generated 85 tokens in 0.942s (90.48 tok/s)
[math_2] Done.
  -> [math_2] Generated 64 tokens

In [15]:
dataset["response_gptq"] = run_inference(MODEL_GPTQ, dataset, "LLAMA_3_2_3B_GPTQ_CUSTOM")


=== BENCHMARKING MODEL: LLAMA_3_2_3B_GPTQ_CUSTOM ===
[Baseline] GPU Allocated: 0.000 GB | CPU Resident: 0.936 GB

=== Loading Model: LLAMA_3_2_3B_GPTQ_CUSTOM (./models/llama-3.2-3b-instruct-gptq-custom) ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


...

Loading weights:   0%|          | 0/842 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 120 objects in 0.230s                             


[Loaded] Model Load Time: 3.10 seconds
[Loaded] Measured Model Weight Size: 2.088 GB
[Loaded] Current GPU Allocated: 2.088 GB | GPU Reserved: 2.156 GB
-> Model Size in Memory: 2.09 GB
Running inference on 10 samples...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[simple_qa_1] Done.
  -> [simple_qa_1] Generated 8 tokens in 0.336s (79.46 tok/s)
[simple_qa_2] Done.
  -> [simple_qa_2] Generated 13 tokens in 0.127s (104.68 tok/s)
[reasoning_1] Done.
  -> [reasoning_1] Generated 83 tokens in 0.788s (105.80 tok/s)
[reasoning_2] Done.
  -> [reasoning_2] Generated 128 tokens in 1.192s (107.92 tok/s)
[coding_1] Done.
  -> [coding_1] Generated 200 tokens in 1.870s (107.13 tok/s)
[coding_2] Done.
  -> [coding_2] Generated 128 tokens in 1.192s (107.71 tok/s)
[summarization] Done.
  -> [summarization] Generated 29 tokens in 0.272s (109.37 tok/s)
[long_context] Done.
  -> [long_context] Generated 96 tokens in 0.925s (104.89 tok/s)
[math_1] Done.
  -> [math_1] Generated 85 tokens in 0.796s (107.25 tok/s)
[math_2] Done.
  -> [math_2] Generated 64 tokens in 0.613s (104.93 tok/s)

[Post-Inference Summary]
[*] Max Peak GPU Memory During Run: 2.124 GB
[*] Total Execution VRAM Overhead (Peak - Baseline): 2.124 GB
[*] Average Generation Latency: 0.811 seconds
[Clean

In [7]:
# 4. Display or save results
print("\n=== Sample Comparison ===")
for idx, row in dataset.head(10).iterrows():
    print(f"\nPrompt Name: {row['name']}")
    print(f"Prompt: {row['prompt']}")
    print(f"-> FP16 Response: {row['response_fp16']}")
    print(f"-> AWQ Response: {row['response_awq']}")


=== Sample Comparison ===

Prompt Name: simple_qa_1
Prompt: What is the capital of Japan?
-> FP16 Response: The capital of Japan is Tokyo.
-> AWQ Response: The capital of Japan is Tokyo.

Prompt Name: simple_qa_2
Prompt: Who wrote the play 'Hamlet'?
-> FP16 Response: The play "Hamlet" was written by William Shakespeare.
-> AWQ Response: The play "Hamlet" was written by William Shakespeare.

Prompt Name: reasoning_1
Prompt: If a train travels 60 km in 45 minutes, what is its average speed in km/h? Explain.
-> FP16 Response: To find the average speed of the train, we need to convert the time from minutes to hours.

Time in hours = 45 minutes / 60 = 0.75 hours

Now, we can use the formula:

Average Speed = Total Distance / Total Time
= 60 km / 0.75 hours
= 80 km/h

So, the average speed of the train is 80 km/h.
-> AWQ Response: To find the average speed of the train, we need to convert the time from minutes to hours.

Time = 45 minutes = 45/60 hours = 0.75 hours

Distance = 60 km

Averag